# AMPS II · Week 03 — HW3
## Thermal stress in 2D, generalized plane strain with bending, a fully coupled transient, and an LLM-generated coupled script

**Due before Week 4 · 100 points.** Submit the executed notebook: *Runtime → Restart session and run all*, fill in the AI-usage log at the end.
Every number you report must come from a cell of this notebook. The first cell of Q4 runs an LLM-generated script that is **meant** to be wrong.

| Q | Topic | Points |
|---|---|:---:|
| 1 | Thermal stress in the 2D reductions, stress-free temperature fields, the coupled 1D diffusion (theory) | 20 |
| 2 | GPS with bending: $\varepsilon_{zz}=e_0+\kappa_xx+\kappa_yy$, a linear temperature field, a bimetal cross-section | 35 |
| 3 | Fully coupled transient: the coupling number, real metals, what a steady test cannot see | 25 |
| 4 | Debugging an LLM-generated coupled thermoelastic script | 20 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

The model modules from the lab (unchanged). Run both cells once.

In [ ]:
%%writefile /tmp/fnx/gps.py
"""2D reductions of a long prism with ONE 3D law. The cross-section is the (x, y) plane, z is the axis of the prism.
eps_zz = 0 (plane strain), eps_zz = e0 with int sigma_zz dA = N (generalized plane strain, GPS), sigma_zz = 0 (plane stress).
Thermal strain alpha*dT; steady heat conduction; meshes: a mapped quarter annulus with curved cells, rectangles."""
import json, os
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, assemble_matrix, assemble_vector, apply_lifting, set_bc

comm = MPI.COMM_WORLD
os.makedirs("/tmp/fnx", exist_ok=True)
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

INNER, OUTER, BOTTOM, LEFT, RIGHT, TOP = 1, 2, 3, 4, 5, 6

def annulus_mesh(ri, ro, nr, nt):
    """quarter annulus ri <= r <= ro, 0 <= theta <= pi/2: nr x nt cells in (r, theta), each split in two triangles,
    6-node (P2) geometry mapped from (r, theta) -> curved arcs, and the same nodes on every machine (no Gmsh).
    Facet tags: 1 r = ri, 2 r = ro, 3 y = 0, 4 x = 0."""
    if comm.rank == 0:
        nI, nJ = 2 * nr + 1, 2 * nt + 1
        R, TH = np.meshgrid(np.linspace(ri, ro, nI), np.linspace(0, np.pi / 2, nJ), indexing="ij")
        x = np.c_[(R * np.cos(TH)).ravel(), (R * np.sin(TH)).ravel()]
        x[np.abs(x) < 1e-14] = 0.0
        node = lambda p: p[0] * nJ + p[1]
        mid = lambda p, q: ((p[0] + q[0]) // 2, (p[1] + q[1]) // 2)
        cells = []
        for a in range(nr):
            for b in range(nt):
                i, j = 2 * a, 2 * b
                for p, q, s in (((i, j), (i + 2, j), (i + 2, j + 2)), ((i, j), (i + 2, j + 2), (i, j + 2))):
                    cells.append([node(p), node(q), node(s), node(mid(q, s)), node(mid(p, s)), node(mid(p, q))])  # basix order
        cells = np.array(cells, dtype=np.int64)
    else:
        x, cells = np.zeros((0, 2)), np.zeros((0, 6), dtype=np.int64)
    msh = mesh.create_mesh(comm, cells, ufl.Mesh(basix.ufl.element("Lagrange", "triangle", 2, shape=(2,))), x)
    rad = lambda X: np.hypot(X[0], X[1])
    return msh, _tags(msh, [(INNER, lambda X: np.isclose(rad(X), ri)), (OUTER, lambda X: np.isclose(rad(X), ro)),
                            (BOTTOM, lambda X: np.isclose(X[1], 0.0)), (LEFT, lambda X: np.isclose(X[0], 0.0))])

def rect_mesh(L, H, nx, ny, x0=0.0, y0=0.0):
    """rectangle [x0, x0+L] x [y0, y0+H], triangles. Facet tags: 3 bottom, 4 left, 5 right, 6 top"""
    msh = mesh.create_rectangle(comm, [[x0, y0], [x0 + L, y0 + H]], [nx, ny], cell_type=mesh.CellType.triangle)
    return msh, _tags(msh, [(BOTTOM, lambda X: np.isclose(X[1], y0)), (LEFT, lambda X: np.isclose(X[0], x0)),
                            (RIGHT, lambda X: np.isclose(X[0], x0 + L)), (TOP, lambda X: np.isclose(X[1], y0 + H))])

def _tags(msh, marks):
    fac, val = [], []
    for tag, f in marks:
        e = mesh.locate_entities_boundary(msh, 1, f); fac.append(e); val.append(np.full(len(e), tag, dtype=np.int32))
    fac, idx = np.unique(np.concatenate(fac), return_index=True)
    return mesh.meshtags(msh, 1, fac, np.concatenate(val)[idx])

def area(msh):
    return comm.allreduce(fem.assemble_scalar(fem.form(1.0 * ufl.dx(domain=msh))), op=MPI.SUM)

def lame_constants(E, nu):
    return E / (2 * (1 + nu)), E * nu / ((1 + nu) * (1 - 2 * nu))

class Material:
    """isotropic with thermal expansion alpha; beta = (3 lambda + 2 mu) alpha is the 3D thermal stress modulus.
    Material(E, nu, alpha) for one material; Material.regions(msh, parts) for piecewise-constant data (DG0 functions)."""
    def __init__(self, E=1.0, nu=0.3, alpha=0.0):
        self.E, self.nu, self.alpha = E, nu, alpha
        self.mu, self.lmbda = lame_constants(E, nu)
        self.beta = (3 * self.lmbda + 2 * self.mu) * alpha

    @staticmethod
    def regions(msh, parts):
        """parts = [(marker(x) -> bool, evaluated at cell midpoints, E, nu, alpha), ...]; later parts overwrite earlier ones"""
        Q = fem.functionspace(msh, ("DG", 0)); m = Material()
        n = msh.topology.index_map(2).size_local + msh.topology.index_map(2).num_ghosts
        mid = mesh.compute_midpoints(msh, 2, np.arange(n, dtype=np.int32)).T
        f = {k: fem.Function(Q) for k in ("E", "nu", "alpha", "mu", "lmbda", "beta")}
        for marker, E, nu, alpha in parts:
            dofs = Q.dofmap.list[np.flatnonzero(marker(mid))].ravel()
            mu, lm = lame_constants(E, nu)
            for k, val in (("E", E), ("nu", nu), ("alpha", alpha), ("mu", mu), ("lmbda", lm), ("beta", (3 * lm + 2 * mu) * alpha)):
                f[k].x.array[dofs] = val
        for k, fun in f.items():
            setattr(m, k, fun)
        return m

def eps3(u, ezz):
    """3D strain of the cross-section: in-plane part from u = (u_x, u_y), eps_zz = ezz, no z-shear"""
    g = (u[0].dx(1) + u[1].dx(0)) / 2
    return ufl.as_matrix([[u[0].dx(0), g, 0], [g, u[1].dx(1), 0], [0, 0, ezz]])

def sigma3(m, u, ezz, dT=0.0):
    """the 3D law  sigma = 2 mu eps + lambda tr(eps) I - beta dT I  (all three reductions use it)"""
    e = eps3(u, ezz)
    return 2 * m.mu * e + m.lmbda * ufl.tr(e) * ufl.Identity(3) - m.beta * dT * ufl.Identity(3)

def ezz_of(m, mode, u, dT, e0):
    """a reduction is a choice of eps_zz: plane strain 0; GPS the unknown number e0; plane stress the value that makes sigma_zz = 0"""
    if mode == "strain":
        return 0.0
    if mode == "gps":
        return e0
    return (m.beta * dT - m.lmbda * ufl.div(u)) / (m.lmbda + 2 * m.mu)

def solve_2d(msh, ft, mat, deg=2, mode="gps", N=0.0, tractions=(), body=None, dT=0.0, fixed=(), points=()):
    """mode 'strain' | 'stress' | 'gps'.  tractions [(tag, (tx, ty))] (numbers or UFL); body force (fx, fy) or None; dT temperature change (number, UFL or Function);
    fixed [(tag, comp)]: u_comp = 0 on those facets; points [(marker, comp)]: u_comp = 0 at the vertices where marker is True.
    GPS by STATIC CONDENSATION of e0: case 1 (loads on, e0 = 0) gives u0 and N0 = int sigma_zz; case 2 (loads off, e0 = 1) gives u1 and N1.
    Both cases have the SAME matrix: one LU factorisation, two solves.  N0 + e0 N1 = N gives e0, and u = u0 + e0 u1.
    Returns uh and info: e0, N0, N1, N (check), sigma (3x3 UFL), the matrix A and the two right-hand sides rhs (for checks)."""
    V = fem.functionspace(msh, ("Lagrange", deg, (2,)))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    ds = ufl.Measure("ds", domain=msh, subdomain_data=ft); dx = ufl.Measure("dx", domain=msh)
    lf, e0c = fem.Constant(msh, 1.0), fem.Constant(msh, 0.0)          # load factor and e0 of the current case
    zero = fem.Constant(msh, np.zeros(2))
    S2 = lambda S: ufl.as_matrix([[S[0, 0], S[0, 1]], [S[1, 0], S[1, 1]]])
    ezz_u = ezz_of(mat, mode, u, 0.0, 0.0) if mode == "stress" else 0.0     # the part of eps_zz that depends on u
    a = ufl.inner(S2(sigma3(mat, u, ezz_u)), ufl.sym(ufl.grad(v))) * dx
    L = ufl.dot(zero, v) * dx - ufl.inner(S2(sigma3(mat, zero, ezz_of(mat, mode, zero, lf * dT, e0c), lf * dT)), ufl.sym(ufl.grad(v))) * dx
    for tag, t in tractions:
        L += lf * ufl.dot(ufl.as_vector(t), v) * ds(tag)
    if body is not None:
        L += lf * ufl.dot(ufl.as_vector(body), v) * dx
    bcs = [fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(c), 1, ft.find(tag)), V.sub(c)) for tag, c in fixed]
    for marker, c in points:
        bcs.append(fem.dirichletbc(0.0, fem.locate_dofs_topological(V.sub(c), 0, mesh.locate_entities_boundary(msh, 0, marker)), V.sub(c)))
    a, L = fem.form(a), fem.form(L)
    A = assemble_matrix(a, bcs=bcs); A.assemble()
    ksp = PETSc.KSP().create(comm); ksp.setOperators(A); ksp.setType("preonly")
    ksp.getPC().setType("lu"); ksp.getPC().setFactorSolverType("mumps")
    rhs = []
    def solve(lf_, e0_):
        lf.value, e0c.value = lf_, e0_
        b = assemble_vector(L); apply_lifting(b, [a], [bcs])
        b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE); set_bc(b, bcs); rhs.append(b)
        w = fem.Function(V); ksp.solve(b, w.x.petsc_vec); w.x.scatter_forward()
        return w
    Nz = lambda w, e0_, lf_: comm.allreduce(fem.assemble_scalar(fem.form(
        sigma3(mat, w, ezz_of(mat, mode, w, lf_ * dT, e0_), lf_ * dT)[2, 2] * dx)), op=MPI.SUM)
    uh = solve(1.0, 0.0)
    info = dict(e0=0.0, A=A, rhs=rhs, reason=ksp.getConvergedReason(), dofs=V.dofmap.index_map.size_global * 2, V=V, ds=ds, dx=dx)
    if mode == "gps":
        u1 = solve(0.0, 1.0)
        N0, N1 = Nz(uh, 0.0, 1.0), Nz(u1, 1.0, 0.0)
        info.update(e0=(N - N0) / N1, N0=N0, N1=N1)
        uh.x.array[:] += info["e0"] * u1.x.array
    info["N"] = Nz(uh, info["e0"], 1.0)
    info["sigma"] = sigma3(mat, uh, ezz_of(mat, mode, uh, dT, info["e0"]), dT)
    return uh, info

def heat(msh, ft, k=1.0, Q=0.0, dirichlet=(), deg=1):
    """steady  -div(k grad T) = Q,  T = value on the tagged facets [(tag, value)], insulated elsewhere"""
    VT = fem.functionspace(msh, ("Lagrange", deg))
    T, s = ufl.TrialFunction(VT), ufl.TestFunction(VT)
    dx = ufl.Measure("dx", domain=msh)
    bcs = [fem.dirichletbc(fem.Constant(msh, float(val)), fem.locate_dofs_topological(VT, 1, ft.find(tag)), VT) for tag, val in dirichlet]
    pr = LinearProblem(k * ufl.inner(ufl.grad(T), ufl.grad(s)) * dx, Q * s * dx + fem.Constant(msh, 0.0) * s * dx, bcs=bcs,
                       petsc_options_prefix="heat_", petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"})
    return pr.solve()

def von_mises(S):
    s = ufl.dev(S)
    return ufl.sqrt(1.5 * ufl.inner(s, s))

def dg_max(msh, expr, deg=1):
    """maximum of a scalar UFL expression over the domain, from its DG(deg) interpolant (call on all ranks)"""
    W = fem.functionspace(msh, ("DG", deg)); f = fem.Function(W)
    f.interpolate(fem.Expression(expr, W.element.interpolation_points))
    return comm.allreduce(float(f.x.array.max()) if f.x.array.size else -np.inf, op=MPI.MAX)

def l2(msh, expr):
    return np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form(ufl.inner(expr, expr) * ufl.dx(domain=msh))), op=MPI.SUM))

In [ ]:
%%writefile /tmp/fnx/coupled.py
"""Transient linear thermoelasticity in ONE mixed space (theta, u): theta = T - T0 (temperature change), plane strain, implicit Euler.
heat:  rho_c d(theta)/dt + cpl * T0 beta d(tr eps)/dt - div(k grad theta) = 0      cpl = 1 full coupling, cpl = 0 one-way (weak)
mech:  div sigma = 0,   sigma = 2 mu eps + lambda tr(eps) I - beta theta I          quasi-static
Strip [0, L] x [0, H]: theta = dT at x = 0, insulated elsewhere; u_x = 0 at x = 0, u_y = 0 on y = 0 and y = H (uniaxial strain),
x = L traction-free. The exact 1D solution has the diffusivity kappa = k / rho_c (one-way) or kappa / (1 + delta) (full)."""
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import assemble_matrix, assemble_vector, apply_lifting, set_bc

comm = MPI.COMM_WORLD
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def moduli(E, nu, alpha):
    mu = E / (2 * (1 + nu)); lmbda = E * nu / ((1 + nu) * (1 - 2 * nu))
    return mu, lmbda, (3 * lmbda + 2 * mu) * alpha

def coupling_delta(E, nu, alpha, rho_c, T0):
    """thermoelastic coupling number  delta = T0 beta^2 / (rho_c (lambda + 2 mu))"""
    mu, lmbda, beta = moduli(E, nu, alpha)
    return T0 * beta**2 / (rho_c * (lmbda + 2 * mu))

def alpha_for_delta(delta, E, nu, rho_c, T0):
    """the expansion coefficient that gives a chosen delta (a 'what if' material)"""
    mu, lmbda, b1 = moduli(E, nu, 1.0)
    return np.sqrt(delta * rho_c * (lmbda + 2 * mu) / T0) / b1

def strip_exact(t, kappa, x=1.0, L=1.0, nterms=400):
    """exact theta(x, t) / dT of  theta_t = kappa theta_xx,  theta(0, t) = dT,  theta_x(L, t) = 0,  theta(x, 0) = 0"""
    n = np.arange(nterms); lam = (2 * n + 1) * np.pi / (2 * L)
    t = np.atleast_1d(np.asarray(t, dtype=float))[:, None]
    return 1 - np.sum(4 / ((2 * n + 1) * np.pi) * np.sin(lam * x) * np.exp(-lam**2 * kappa * t), axis=1)

def t_half(t, y):
    """first time at which y reaches 1/2 (linear interpolation between steps)"""
    i = int(np.argmax(np.asarray(y) >= 0.5))
    return t[i - 1] + (0.5 - y[i - 1]) * (t[i] - t[i - 1]) / (y[i] - y[i - 1])

def run_strip(E=1.0, nu=0.3, alpha=0.1, k=1.0, rho_c=1.0, T0=1.0, dT=1.0, cpl=1.0, L=1.0, H=0.1, nx=64, ny=2,
              dt=2e-3, t_end=1.5, degT=1, degu=2):
    """returns dict: t, TL = mean theta(L, t)/dT, UL = mean u_x(L, t), delta, kappa, beta, reason, and the final w (theta, u)"""
    msh = mesh.create_rectangle(comm, [[0, 0], [L, H]], [nx, ny], cell_type=mesh.CellType.triangle)
    el = basix.ufl.mixed_element([basix.ufl.element("Lagrange", "triangle", degT),
                                  basix.ufl.element("Lagrange", "triangle", degu, shape=(2,))])
    W = fem.functionspace(msh, el)                                  # ONE space: sub(0) = theta, sub(1) = u
    mu, lmbda, beta = moduli(E, nu, alpha)
    (th, u), (s, v) = ufl.TrialFunctions(W), ufl.TestFunctions(W)
    w_n = fem.Function(W); th_n, u_n = ufl.split(w_n)               # previous step
    eps = lambda w: ufl.sym(ufl.grad(w))
    sig = lambda w, T: 2 * mu * eps(w) + lmbda * ufl.tr(eps(w)) * ufl.Identity(2) - beta * T * ufl.Identity(2)   # plane strain
    idt, c = fem.Constant(msh, 1.0 / dt), fem.Constant(msh, float(cpl))
    a = (rho_c * idt * th * s + c * T0 * beta * idt * ufl.tr(eps(u)) * s + k * ufl.inner(ufl.grad(th), ufl.grad(s))) * ufl.dx \
        + ufl.inner(sig(u, th), eps(v)) * ufl.dx
    Lf = (rho_c * idt * th_n * s + c * T0 * beta * idt * ufl.tr(eps(u_n)) * s) * ufl.dx
    fd = lambda f: mesh.locate_entities_boundary(msh, 1, f)
    left = fd(lambda X: np.isclose(X[0], 0)); tb = fd(lambda X: np.isclose(X[1], 0) | np.isclose(X[1], H))
    bcs = [fem.dirichletbc(fem.Constant(msh, float(dT)), fem.locate_dofs_topological(W.sub(0), 1, left), W.sub(0)),
           fem.dirichletbc(0.0, fem.locate_dofs_topological(W.sub(1).sub(0), 1, left), W.sub(1).sub(0)),
           fem.dirichletbc(0.0, fem.locate_dofs_topological(W.sub(1).sub(1), 1, tb), W.sub(1).sub(1))]
    a, Lf = fem.form(a), fem.form(Lf)
    A = assemble_matrix(a, bcs=bcs); A.assemble()                   # constant in time: factorise once
    ksp = PETSc.KSP().create(comm); ksp.setOperators(A); ksp.setType("preonly")
    ksp.getPC().setType("lu"); ksp.getPC().setFactorSolverType("mumps")
    right = np.sort(fd(lambda X: np.isclose(X[0], L)))
    dsR = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, 1, right, np.full(len(right), 1, np.int32)))
    thL, uL = fem.form(w_n.sub(0) * dsR(1)), fem.form(w_n.sub(1)[0] * dsR(1))
    w = fem.Function(W); ts, TL, UL = [0.0], [0.0], [0.0]
    for n in range(1, int(round(t_end / dt)) + 1):
        b = assemble_vector(Lf); apply_lifting(b, [a], [bcs])
        b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE); set_bc(b, bcs)
        ksp.solve(b, w.x.petsc_vec); w.x.scatter_forward()
        w_n.x.array[:] = w.x.array
        ts.append(n * dt)
        TL.append(comm.allreduce(fem.assemble_scalar(thL), op=MPI.SUM) / H / dT)
        UL.append(comm.allreduce(fem.assemble_scalar(uL), op=MPI.SUM) / H)
    return dict(t=np.array(ts), TL=np.array(TL), UL=np.array(UL), delta=T0 * beta**2 / (rho_c * (lmbda + 2 * mu)),
                kappa=k / rho_c, beta=beta, lmbda=lmbda, mu=mu, reason=ksp.getConvergedReason(), w=w, W=W, msh=msh)

## Q1 — Theory (20 pts)

**(a) (7)** Start from $\boldsymbol\sigma=2\mu\boldsymbol\varepsilon+\lambda(\operatorname{tr}\boldsymbol\varepsilon)\mathbf I-\beta\Delta T\,\mathbf I$, $\beta=(3\lambda+2\mu)\alpha$.
Derive the in-plane law and $\sigma_{zz}$ for plane strain, and for plane stress (eliminate $\varepsilon_{zz}$ with $\sigma_{zz}=0$). Show that the plane-stress
thermal modulus is $E\alpha/(1-\nu)$, and that $\beta=E\alpha/(1-2\nu)$. What happens to each as $\nu\to\tfrac12$?

**(b) (7)** In plane stress, a free simply connected plate is stress-free iff the strain $\alpha\Delta T\,\mathbf I$ is compatible. Use the 2D compatibility
condition $\partial_{yy}\varepsilon_{xx}+\partial_{xx}\varepsilon_{yy}=2\partial_{xy}\varepsilon_{xy}$ to show that this holds iff $\nabla^2\Delta T=0$.
For a harmonic $\Delta T$ give $\sigma_{zz}$ in (i) plane strain, (ii) GPS with $N=0$ ($\varepsilon_{zz}=e_0$), (iii) GPS with $\varepsilon_{zz}=e_0+\kappa_xx+\kappa_yy$.
Which temperature fields make (iii) completely stress-free? (Q2(a) tests this.)

**(c) (6)** Strip in uniaxial strain ($u_y=0$, $\varepsilon_{zz}=0$), free end so $\sigma_{xx}=0$, no body force. From the two field equations of lecture §A7
derive $\rho c(1+\delta)\dot\theta=k\theta''$ with $\delta=T_0\beta^2/(\rho c(\lambda+2\mu))$. Predict the ratio of the half-rise times $t_{50}$ (full / one-way) and
what happens if the sign of the coupling term is flipped.

> *your answer*

## Q2 — Generalized plane strain with bending (35 pts)

GPS with only $e_0$ cannot represent a prism that **bends** about its own axis. Allow $\varepsilon_{zz}=e_0+\kappa_xx+\kappa_yy$ (three unknown numbers) with
three equations: $\int\sigma_{zz}\,dA=N$, $\int\sigma_{zz}\,x\,dA=M_x$, $\int\sigma_{zz}\,y\,dA=M_y$ (here all zero: free ends).

**(a) (12)** Write `solve_gps_bend(msh, ft, mat, deg, NM, dT, fixed, points)` by static condensation: **four** solves with one matrix (the load case and the
three unit cases), then a $3\times3$ system. Start from `solve_2d` in `gps.py`. Test it on the homogeneous rectangle $[-\tfrac12,\tfrac12]\times[-\tfrac14,\tfrac14]$
($E=1$, $\nu=0.3$, $\alpha=10^{-3}$, pin + roller) with $\Delta T=10+30x+20y$: print $(e_0,\kappa_x,\kappa_y)$, and the maximum von Mises stress for plane strain,
GPS with $e_0$ only, and GPS with bending. Compare with Q1(b).

In [ ]:
%%writefile /tmp/fnx/bend.py
# TODO: Q2(a) — solve_gps_bend(msh, ft, mat, deg=2, NM=(0, 0, 0), dT=0.0, fixed=(), points=()) -> (uh, info)
#       start from solve_2d in gps.py: four right-hand sides with one KSP, then a 3x3 system for (e0, kx, ky)
from gps import *

In [ ]:
%%fenicsx -np 2
# TODO: Q2(a) — the linear-temperature test of solve_gps_bend

> *your comparison with Q1(b) (Q2a)*

**(b) (13)** Bimetal cross-section: steel ($E_1=210$ GPa, $\alpha_1=1.2\times10^{-5}$) for $y<0$, aluminium ($E_2=70$ GPa, $\alpha_2=2.3\times10^{-5}$) for $y>0$,
total height $h=1$ (equal layers), width $w$, **both $\nu=0.3$**, uniform $\Delta T=100$ K, free ends. Use `Material.regions`. For $w=0.1,\,1,\,10$
(about 40 cells over the height, P2) compute $\kappa_y$ and compare with Timoshenko's bimetal formula (beam theory, $m=h_1/h_2$, $n=E_1/E_2$)
$$
\kappa_T=\frac{6(\alpha_2-\alpha_1)\Delta T\,(1+m)^2}{h\left[3(1+m)^2+(1+mn)\left(m^2+\dfrac1{mn}\right)\right]} .
$$
Beam theory ignores the in-plane stresses that the mismatch causes. Explain what you find. *(Hint: in a traction-free, simply connected section the
in-plane stress has an Airy function $\phi$ with $\phi=\partial\phi/\partial n=0$ on the boundary and $\sigma_{xx}+\sigma_{yy}=\nabla^2\phi$. What is
$\int(\sigma_{xx}+\sigma_{yy})\,w\,dA$ for a linear $w$? Where does $\nu$ enter $\sigma_{zz}$?)*

In [ ]:
%%fenicsx -np 2
# TODO: Q2(b) — the bimetal section for three widths, kappa_y / kappa_T

> *your results and explanation (Q2b)*

**(c) (10)** Now give each layer its own Poisson ratio: steel 0.30, aluminium 0.33. Repeat (b), and for $w=1$ also on a mesh refined twice in each
direction. (i) How far is $\kappa_y$ from $\kappa_T$, and how does that depend on $w$? (ii) What does the maximum von Mises stress do under refinement, and
where is it? (iii) Which numbers of this model would you report to a designer, and which not?

In [ ]:
%%fenicsx -np 2
# TODO: Q2(c) — unequal Poisson ratios: kappa_y / kappa_T vs width, and the peak stress under refinement

> *your answers (Q2c)*

## Q3 — The fully coupled transient (25 pts)

Use `run_strip` from `coupled.py` (lab §5): $E=1$, $\nu=0.3$, $\rho c=1$, $k=1$, $T_0=1$, $\alpha$ = `alpha_for_delta(delta, 1.0, 0.3, 1.0, 1.0)`, $\Delta t=2\times10^{-3}$.

**(a) (10)** For $\delta=0.1,\,0.25,\,0.5,\,1.0$ compute $t_{50}$ of the full model and the ratio to the one-way $t_{50}$ (the one-way run does not depend on $\delta$ —
why?). Table and plot of the ratio against $\delta$ with the prediction of Q1(c).

**(b) (8)** Steel and aluminium (data in lab §5 and lecture §A8): compute $\delta$ with `coupling_delta`, run the strip with the same $\delta$, and report the ratio.
How large is the error of a one-way model for these metals, and is it larger or smaller than the time-step error of this computation?

**(c) (7)** Flip the sign of the coupling term (`cpl=-1.0`) at $\delta=0.5$, run to $t=4$, and compare with `cpl=+1.0`: $t_{50}$, and $\theta(L)$ and $u_x(L)$ at the end.
What would a steady-state comparison of "full" and "one-way" have told you about this sign error?

In [ ]:
%%fenicsx -np 2
# TODO: Q3(a) — delta sweep

In [ ]:
# TODO: Q3(a) — plot of the ratio against delta

> *your table and comment (Q3a)*

In [ ]:
%%fenicsx -np 2
# TODO: Q3(b) — steel and aluminium

> *your answer (Q3b)*

In [ ]:
%%fenicsx -np 2
# TODO: Q3(c) — the flipped sign

> *your answer (Q3c)*

## Q4 — Debugging an LLM-generated coupled thermoelastic script (20 pts)

An LLM was asked for "a fully coupled plane-strain thermoelastic model of a strip $[0,1]\times[0,0.1]$, heated by 1 K at $x=0$ from $T_0=300$ K, insulated
elsewhere, $u_x=0$ at $x=0$, $u_y=0$ on top and bottom, free at $x=1$, with $E=1$, $\nu=0.3$, $k=\rho c=1$ and $\alpha$ chosen for $\delta\approx0.5$; compare the
half-rise time at $x=1$ with a one-way model; the ratio should be $1+\delta$". The cell below is what it produced. It contains **four** defects.

**(a) (6)** Run it. Without looking for the defects yet, list the outputs that **cannot** be right and the physical argument for each.
**(b) (8)** Find all four defects (what is wrong, how you found it) and write the corrected script in the cell after it.
**(c) (6)** Verify the corrected script against the exact series (`strip_exact` from `coupled.py`): $t_{50}$ of both runs, $\theta(L)$ at the end, the ratio.

In [ ]:
%%fenicsx -np 2
# LLM-generated script (HW3 Q4) — it runs, but it contains four defects
from mpi4py import MPI
import numpy as np, ufl, basix.ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD
L, H, nx = 1.0, 0.1, 64
E, nu, k, rho_c = 1.0, 0.3, 1.0, 1.0
T0, T_hot = 300.0, 301.0                         # reference and heated-end temperature [K]
alpha = 0.01895                                  # chosen so that delta is about 0.5
mu = E / (2 * (1 + nu)); lmbda = E * nu / ((1 + nu) * (1 - 2 * nu))
beta = E * alpha / (1 - nu)                      # thermal stress modulus
delta = T0 * beta**2 / (rho_c * (lmbda + 2 * mu))

msh = mesh.create_rectangle(comm, [[0, 0], [L, H]], [nx, 2], cell_type=mesh.CellType.triangle)
W = fem.functionspace(msh, basix.ufl.mixed_element([basix.ufl.element("Lagrange", "triangle", 1),
                                                     basix.ufl.element("Lagrange", "triangle", 2, shape=(2,))]))
V0, _ = W.sub(0).collapse()
xT = V0.tabulate_dof_coordinates()[:V0.dofmap.index_map.size_local, 0]   # owned temperature DOFs
atL = np.isclose(xT, L)

def run(coupled, dt=4e-3, t_end=1.2):
    (T, u), (s, v) = ufl.TrialFunctions(W), ufl.TestFunctions(W)
    w_n = fem.Function(W); w_n.sub(0).interpolate(lambda x: np.full(x.shape[1], T0))   # start at T0
    T_n, u_n = ufl.split(w_n)
    eps = lambda w: ufl.sym(ufl.grad(w))
    sig = lambda w, T: 2 * mu * eps(w) + lmbda * ufl.tr(eps(w)) * ufl.Identity(2) - beta * T * ufl.Identity(2)
    c = 1.0 if coupled else 0.0
    a = (rho_c / dt * T * s + c * T0 * beta / dt * ufl.tr(eps(u)) * s + k * ufl.inner(ufl.grad(T), ufl.grad(s))) * ufl.dx \
        + ufl.inner(sig(u, T), eps(v)) * ufl.dx
    Lf = rho_c / dt * T_n * s * ufl.dx
    left = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[0], 0))
    tb = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[1], 0) | np.isclose(x[1], H))
    bcs = [fem.dirichletbc(fem.Constant(msh, T_hot), fem.locate_dofs_topological(W.sub(0), 1, left), W.sub(0)),
           fem.dirichletbc(0.0, fem.locate_dofs_topological(W.sub(1).sub(0), 1, left), W.sub(1).sub(0)),
           fem.dirichletbc(0.0, fem.locate_dofs_topological(W.sub(1).sub(1), 1, tb), W.sub(1).sub(1))]
    problem = LinearProblem(a, Lf, bcs=bcs, petsc_options_prefix="te_",
                            petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"})
    ts, TL = [0.0], [0.0]
    for n in range(1, int(round(t_end / dt)) + 1):
        w = problem.solve()
        w_n.x.array[:] = w.x.array
        T_vals = w.x.array[:len(xT)]                   # temperature values
        TmL = comm.allreduce(T_vals[atL].sum(), op=MPI.SUM) / comm.allreduce(int(atL.sum()), op=MPI.SUM)
        TL.append((TmL - T0) / (T_hot - T0)); ts.append(n * dt)
    return np.array(ts), np.array(TL), w

def t_half(t, y):
    i = int(np.argmax(y >= 0.5)); return t[i - 1] + (0.5 - y[i - 1]) * (t[i] - t[i - 1]) / (y[i] - y[i - 1])

t, TLw, _ = run(False)
t, TLf, w = run(True)
uL = comm.allreduce(w.sub(1).collapse().x.array.reshape(-1, 2)[:, 0].max(), op=MPI.MAX)
if comm.rank == 0:
    print(f"delta = {delta:.4f}")
    print(f"t50 one-way = {t_half(t, TLw):.4f}   t50 coupled = {t_half(t, TLf):.4f}   ratio = {t_half(t, TLf) / t_half(t, TLw):.4f}"
          f"   (theory 1 + delta = {1 + delta:.4f})")
    print(f"theta(L)/dT at t = 1.2: one-way {TLw[-1]:.4f}, coupled {TLf[-1]:.4f};  u_x(L) at t = 1.2: {uL:.4e}")

> *your list of impossible outputs (Q4a)*

In [ ]:
%%fenicsx -np 2
# TODO: Q4(b) — the corrected script

> *your list of defects (Q4b)*

In [ ]:
%%fenicsx -np 1
# TODO: Q4(c) — verification against the exact series

> *your verification (Q4c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …